In [2]:
import os
import json

INPUT_FILE = "./results_ablation/remove_phase1_gaze.jsonl" 


def evaluate_and_extract_errors():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 文件不存在: {INPUT_FILE}")
        return

    # 初始化计数器
    stats = {
        "total": 0,
        "correct": 0,
        "tp": 0, # 真阳性 (GT=True, Pred=True)
        "tn": 0, # 真阴性 (GT=False, Pred=False)
        "fp": 0, # 假阳性 (GT=False, Pred=True) -> 误报
        "fn": 0  # 假阴性 (GT=True, Pred=False) -> 漏报
    }
    
    error_cases = []

    print(f"📂 正在读取并修复文件格式: {INPUT_FILE} ...")

    # 1. 读取整个文件内容到内存
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        content = f.read()

    # 2. 使用 raw_decode 逐个解析 JSON 对象
    decoder = json.JSONDecoder()
    pos = 0
    total_len = len(content)

    while pos < total_len:
        # 跳过空白字符 (空格、换行等)
        while pos < total_len and content[pos].isspace():
            pos += 1
        
        if pos >= total_len:
            break

        try:
            # raw_decode 返回 (解析出的对象, 解析结束后的新位置)
            item, end_pos = decoder.raw_decode(content, idx=pos)
            pos = end_pos # 更新位置指针
            
            # ================= 评估逻辑开始 =================
            gt = item.get("ground_truth")
            pred = item.get("prediction")

            # 跳过没有标签的数据 或 预测为空的数据
            if gt is None or pred is None:
                continue

            stats["total"] += 1

            # 确保 pred 是布尔值 (防止模型输出 "true"/"false" 字符串)
            if isinstance(pred, str):
                pred = True if pred.lower() == 'true' else False

            if gt == pred:
                stats["correct"] += 1
                if gt is True: stats["tp"] += 1
                else: stats["tn"] += 1
            else:
                # 记录错误类型
                if pred is True: 
                    stats["fp"] += 1
                    item["error_type"] = "False Positive (误报)"
                else: 
                    stats["fn"] += 1
                    item["error_type"] = "False Negative (漏报)"
                
                error_cases.append(item)
            # ================= 评估逻辑结束 =================

        except json.JSONDecodeError:
            # 如果解析失败，说明这里有非法字符，跳过一个字符尝试继续
            # print(f"⚠️ 解析警告: 在位置 {pos} 跳过非法字符")
            pos += 1

    # --- 计算指标 ---
    if stats["total"] == 0:
        print("❌ 未找到有效数据，请检查 JSON 字段名是否匹配 (ground_truth / sarcasm_prediction)")
        return

    acc = stats["correct"] / stats["total"]
    # 防止除以零
    precision = stats["tp"] / (stats["tp"] + stats["fp"]) if (stats["tp"] + stats["fp"]) > 0 else 0
    recall = stats["tp"] / (stats["tp"] + stats["fn"]) if (stats["tp"] + stats["fn"]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # --- 打印报告 ---
    print(f"\n{'='*30}")
    print(f"📊 评估结果 (样本数: {stats['total']})")
    print(f"{'='*30}")
    print(f"✅ Accuracy (准确率):  {acc:.2%}")
    print(f"🎯 Precision (查准率): {precision:.2%}")
    print(f"🔎 Recall    (查全率): {recall:.2%}")
    print(f"⚖️  F1-Score:         {f1:.2%}")
    print(f"{'-'*30}")
    print(f"详细统计:")
    print(f"  TP (讽刺且预测对): {stats['tp']}")
    print(f"  TN (正常且预测对): {stats['tn']}")
    print(f"  FP (误判为讽刺):   {stats['fp']} (模型太敏感)")
    print(f"  FN (漏判讽刺):     {stats['fn']} (模型没识别出)")
    print(f"{'='*30}")

if __name__ == "__main__":
    evaluate_and_extract_errors()

📂 正在读取并修复文件格式: ./results_ablation/remove_phase1_gaze.jsonl ...

📊 评估结果 (样本数: 14)
✅ Accuracy (准确率):  57.14%
🎯 Precision (查准率): 100.00%
🔎 Recall    (查全率): 33.33%
⚖️  F1-Score:         50.00%
------------------------------
详细统计:
  TP (讽刺且预测对): 3
  TN (正常且预测对): 5
  FP (误判为讽刺):   0 (模型太敏感)
  FN (漏判讽刺):     6 (模型没识别出)
